# SCENARIO 2: STRUCTURED STREAMING
A retail company receives daily sales transaction files from multiple store locations in an Azure Data Lake folder. Instead of reprocessing all historical data every day, the data engineering team uses Spark Structured Streaming to incrementally load only the newly arrived files into a Delta table. This ensures timely updates to analytics dashboards while optimizing compute costs and processing time.

Structured Streaming lets Spark treat the continuously growing folder as a stream of new data, processing newly arrived files incrementally rather than repeatedly scanning all historical data.

## Streaming Query

In [0]:
my_schema = """
order_id INT,
customer_id INT,
order_date DATE,
amount DOUBLE
"""

In [0]:
df_stream = (spark.readStream               # readStream: treat this DIRECTORY as a streaming source and process new data as it arrives
            .format('csv')
            .option('header', True)
            .schema(my_schema)           # always add defined schema in Spark Streaming
            .load('/Volumes/pyspark_catalog/source/db_volume/streamSource/'))          # pass the folder level since we have multiple files in there

## Streaming Output

In [0]:
# After Spark reads the new files, you write the data into a Delta table
output = (df_stream.writeStream
          .format('delta')              # always store your data in open table formats: delta, apache iceberg,... 
          .option('checkpointLocation', '/Volumes/pyspark_catalog/source/db_volume/streamSink/checkpoint')
          .option('mergeSchema', True)
          .trigger(availableNow=True)           # Process all currently available data, possibly in multiple batches, then stop
          .start('/Volumes/pyspark_catalog/source/db_volume/streamSink/data'))          # start <=> load

In [0]:
%sql
SELECT * FROM delta.`/Volumes/pyspark_catalog/source/db_volume/streamSink/data/`

order_id,customer_id,order_date,amount
1,101,2025-08-02,246.84
2,104,2025-08-03,111.3
3,103,2025-08-04,52.0
4,103,2025-08-05,98.7
5,102,2025-08-06,392.67
